# Langchain Implementation

In [3]:
import os

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [2]:
department_mapping = {
    'code_of_conduct.txt': "Admin",
    'it_security_policy.txt': 'IT',
    'leave_policy.txt': "HR",
    'remote_work_policy.txt': "HR"
}

data_folder = "data"

In [4]:
# Step 1: Loading Documents

def load_documents(data_folder):
    documents = []

    for file in os.listdir(data_folder):
        path=os.path.join(data_folder, file)
        
        with open(path, 'r') as f:
            text = f.read()
        
        doc = Document(
            page_content=text,
            metadata={'source': file,
                    'department': department_mapping.get(file, 'Unknown')}
        )    
        documents.append(doc)

    print('Documents:')
    print(documents)
    
    return documents

In [6]:
documents = load_documents(data_folder)

Documents:
[Document(metadata={'source': 'code_of_conduct.txt', 'department': 'Admin'}, page_content='AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.\n\nHarassment, discrimination, or unethical conduct will not be tolerated.'), Document(metadata={'source': 'it_security_policy.txt', 'department': 'IT'}, page_content='AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.'), Document(metadata={'source': 'leave_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\n\nLeave requests must be submitted at least two weeks in advance.\n\nUnused leave may not be carried over to the next year unless approved.'), Document(metadata={'source': 'remote_work_policy.txt', 'depar

In [5]:
# Step 2: Recursive Chunks using Langchain Recursive Chunk Splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 20
)

text_splitter

In [7]:
chunks = text_splitter.split_documents(documents=documents)
chunks

[Document(metadata={'source': 'code_of_conduct.txt', 'department': 'Admin'}, page_content='AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.'),
 Document(metadata={'source': 'code_of_conduct.txt', 'department': 'Admin'}, page_content='Harassment, discrimination, or unethical conduct will not be tolerated.'),
 Document(metadata={'source': 'it_security_policy.txt', 'department': 'IT'}, page_content='AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.'),
 Document(metadata={'source': 'it_security_policy.txt', 'department': 'IT'}, page_content='All systems must be locked when unattended.'),
 Document(metadata={'source': 'it_security_policy.txt', 'department': 'IT'}, page_content='Sensitive data must be stored securely and encrypted where appropriate.'),
 Document(metadata={'source': 'leave_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Leave Policy'),
 Document(metadata={'source':

In [8]:
# Step 3: Create Embeddings

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small', api_key=os.getenv('OPENAI_SECRET_KEY'))
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002D6D9BC9850>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002D6D9B906D0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [10]:
# Step 4: Create Vector Store

vs = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory='./langchain__hbd_db',
    collection_metadata={'hnsw:space': 'cosine'},
    collection_name='langchain_collection'
)

vs

# LECL Retrieval

In [19]:
# LECL - Langchain Expression Language

# We create chain using | operator

# Example
# chain = prompt | llm | parser

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

## Retriever using Cosine Similarity + Threshold

### Step 1: Create Retriever using Cosine Similarity + Threshold

In [15]:
retriever = vs.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={
        'score_threshold': 0.5,
        "k": 5
    }
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002D6DB9C4B90>, search_type='similarity_score_threshold', search_kwargs={'score_threshold': 0.5, 'k': 5})

### Step 2: Create Prompt Template

In [25]:
prompt = ChatPromptTemplate.from_template(
    '''Answer the given query based upon the given context only. 
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    CONTEXT:
    {context}
    QUERY:
    {query}'''
)

prompt

ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, template="Answer the given query based upon the given context only. \n    If you don't know the answer, just say that you don't know, don't try to make up an answer.\n    CONTEXT:\n    {context}\n    QUERY:\n    {query}"), additional_kwargs={})])

### Step 3: Create LLM

In [26]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv('OPENAI_SECRET_KEY'))
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002D6DFEEB550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002D6DC09CD10>, root_client=<openai.OpenAI object at 0x000002D6DFEEB190>, root_async_client=<openai.AsyncOpenAI object at 0x000002D6DF2FE610>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

### Step 4: Format Documents

In [34]:
def format_documents(docs):
    print(docs)
    return '\n\n'.join([doc.page_content for doc in docs])

### Step 5: Create LECL Chain

In [36]:
chain = (
    {'context': retriever | format_documents,
     'query': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002D6DB9C4B90>, search_type='similarity_score_threshold', search_kwargs={'score_threshold': 0.5, 'k': 5})
           | RunnableLambda(format_documents),
  query: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, template="Answer the given query based upon the given context only. \n    If you don't know the answer, just say that you don't know, don't try to make up an answer.\n    CONTEXT:\n    {context}\n    QUERY:\n    {query}"), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 

### Step 6: Invoke Chain

In [37]:
query = 'How many days in a week an employee can work from home?'

response = chain.invoke(query)

response

[Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.'), Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager.")]


'An employee can work from home up to 2 days per week.'

In [30]:
query = 'How many days in a year an employee can work from home?'

response = chain.invoke(query)

response

'An employee can work from home up to 104 days in a year (2 days per week for 52 weeks).'

In [38]:
query = 'Capital of India?'

response = chain.invoke(query)

response

No relevant docs were retrieved using the relevance score threshold 0.5


[]


"I don't know."

In [43]:
query = 'I have a shared password.'

response = chain.invoke(query)

response

No relevant docs were retrieved using the relevance score threshold 0.5


[]


"I don't know."

In [44]:
# Chains can work with Langchain runnables only

# Steps
# Step 1 | Step 2 | Step 3

def dummy():
    pass

RunnableLambda(dummy)

RunnableLambda(dummy)

## Hybrid Search Retriever

### Step 1: Create Retriever (Cosine Similarity + Threshold)

In [45]:
retriever = vs.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={
        'score_threshold': 0.5,
        'k': 5
    }
)

### Step 2: Create Hybrid Retriever (Vector + Keyworded Search)

In [47]:
vs.get()['documents']

['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.',
 'Harassment, discrimination, or unethical conduct will not be tolerated.',
 'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.',
 'All systems must be locked when unattended.',
 'Sensitive data must be stored securely and encrypted where appropriate.',
 'AcmeTech Solutions Leave Policy',
 'All full-time employees are entitled to 20 days of paid annual leave per calendar year.',
 'Leave requests must be submitted at least two weeks in advance.',
 'Unused leave may not be carried over to the next year unless approved.',
 'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.',
 "Remote work must be approved by the employee's manager.",
 'Employees must ensure a secure and productive work environment while working remotely.']

In [64]:
def hybrid_retriever_fn(query):
    # vector retrieval
    vector_docs = retriever.invoke(query)
    print('Vector Docs:')
    print(vector_docs)
    
    # keyworded retrieval
    all_data = vs.get()
    documents = all_data['documents']
    metadatas = all_data['metadatas']
    
    keyworded_docs = []
    for text, metadata in zip(documents, metadatas):
        if query.lower() in text.lower():
            keyworded_docs.append(Document(page_content=text, metadata=metadata))      
          
    print('Keyworded Docs:')
    print(keyworded_docs)  
    # combine and remove duplicates
    combined = {doc.page_content: doc for doc in vector_docs}
    
    for doc in keyworded_docs:
        combined[doc.page_content] = doc
        
    return list(combined.values())  

In [58]:
retriever.invoke('work from home policy')

[Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.'),
 Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='Employees must ensure a secure and productive work environment while working remotely.'),
 Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager.")]

In [65]:
hybrid_retriever_fn('work from home policy')

Vector Docs:
[Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.'), Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='Employees must ensure a secure and productive work environment while working remotely.'), Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager.")]
Keyworded Docs:
[]


[Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.'),
 Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='Employees must ensure a secure and productive work environment while working remotely.'),
 Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager.")]

In [66]:
hybrid_retriever_fn('Remote Work')

Vector Docs:
[Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager."), Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='Employees must ensure a secure and productive work environment while working remotely.'), Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.')]
Keyworded Docs:
[Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.'), Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager.")]


[Document(metadata={'department': 'HR', 'source': 'remote_work_policy.txt'}, page_content="Remote work must be approved by the employee's manager."),
 Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='Employees must ensure a secure and productive work environment while working remotely.'),
 Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.')]

### Step 3: Add LLM with Reranking

In [61]:
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002D6DFEEB550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002D6DC09CD10>, root_client=<openai.OpenAI object at 0x000002D6DFEEB190>, root_async_client=<openai.AsyncOpenAI object at 0x000002D6DF2FE610>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [67]:
def rerank_fn(data):
    query = data['query']
    docs = data['context']
    
    scores = []
    
    for doc in docs:
        prompt=f"""Rate the relevance of this document with the given query.
Always give the rating from 0 to 10.
QUERY:
{query}
DOCUMENT:
{doc.page_content}
"""
        response = llm.invoke(prompt)
        try:
            score = float(response.content.strip())
        except:
            score = 0
            
        scores.append((doc, score))
    
    # sort them in descending order
    scores.sort(key=lambda x:x[1], reverse=True)
    
    # keep top 3
    top_docs = [doc for doc in scores[:3]]
    
    return {
        'query': query,
        'context': top_docs
    }

### Step 4: Create Prompt

In [63]:
prompt = ChatPromptTemplate.from_template(
    '''Answer the given query based upon the given context only. 
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    CONTEXT:
    {context}
    QUERY:
    {query}'''
)

prompt

ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, template="Answer the given query based upon the given context only. \n    If you don't know the answer, just say that you don't know, don't try to make up an answer.\n    CONTEXT:\n    {context}\n    QUERY:\n    {query}"), additional_kwargs={})])

### Step 5: Create LECL Chain/Pipeline

In [70]:
hybrid_retriever = RunnableLambda(hybrid_retriever_fn)
rerank = RunnableLambda(rerank_fn)

hybrid_retriever

RunnableLambda(hybrid_retriever_fn)

In [77]:
chain = ({'query': RunnablePassthrough(),
          'context': hybrid_retriever}
         | rerank
         | {'query': lambda x:x['query'],
            'context': lambda x:x['context']}
         | prompt
         | llm
         | StrOutputParser())

In [78]:
chain

{
  query: RunnablePassthrough(),
  context: RunnableLambda(hybrid_retriever_fn)
}
| RunnableLambda(lambda x: x[1])
| {
    query: RunnableLambda(...),
    context: RunnableLambda(...)
  }
| ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, template="Answer the given query based upon the given context only. \n    If you don't know the answer, just say that you don't know, don't try to make up an answer.\n    CONTEXT:\n    {context}\n    QUERY:\n    {query}"), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': T

### Step 6: Invoke Chain

In [79]:
query = 'How many days in a week an employee can work from home?'

response = chain.invoke(query)

response

Vector Docs:
[Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content='AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.'), Document(metadata={'source': 'remote_work_policy.txt', 'department': 'HR'}, page_content="Remote work must be approved by the employee's manager.")]
Keyworded Docs:
[]


'Employees are allowed to work remotely up to 2 days per week.'

In [81]:
query = 'I am using shared password.'

response = chain.invoke(query)

response

Vector Docs:
[Document(metadata={'source': 'it_security_policy.txt', 'department': 'IT'}, page_content='AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.')]
Keyworded Docs:
[]


"I don't know."

In [82]:
# Langchain Flow
# Simple Retriever
# Cosine+threshold Retriever
# Hybrid Retriever
# Hybrid + Rerank Retriever